# Phase 2E — Balanced Sampling (Side Experiment)

**Hypothesis:** replacing focal loss + class weights with data-level class balancing
(WeightedRandomSampler) and plain cross-entropy gives the model more equal exposure
to minority classes during training.

Everything else is identical to Phase 2B:
- Same RETFound-DINOv2-MEH backbone, full fine-tune
- Same LLRD (decay=0.75), gradient checkpointing, gradient accumulation (2 steps)
- Same 5-fold patient-level stratified CV, same random seed

**What changes vs P2B:**
| | P2B | P2E (this notebook) |
|---|---|---|
| Loss | FocalLoss γ=2 + class weights | CrossEntropyLoss (no weights) |
| Sampling | Random shuffle | WeightedRandomSampler (balanced) |
| Output dir | `output_dir/phase2b_cv/` | `output_dir/phase2e_cv/` |

No existing files are modified.

In [ ]:
import os, sys, json, copy, math, time
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

os.chdir(os.path.dirname(os.path.abspath('phase2e_balanced_sampling.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Shared constants (identical to P2B) ───────────────────────────────────
N_FOLDS      = 5
MAX_EPOCHS   = 50
PATIENCE     = 10
INPUT_SIZE   = 224
NUM_CLASSES  = 4
CLASSES      = ['R0', 'R1', 'R2', 'R3A']
SEED         = 42

BASE_LR       = 5e-5
MIN_LR        = 1e-7
WARMUP_EPOCHS = 5
WEIGHT_DECAY  = 0.05
LLRD_DECAY    = 0.75
GRAD_CLIP     = 1.0
BATCH_SIZE    = 16
ACCUM_STEPS   = 2

HF_REPO   = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE   = 'RETFound_dinov2_meh.pth'
CV_OUTPUT = Path('output_dir/phase2e_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(DEVICE)
    print(f'GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB')
print('Phase 2E — balanced sampling, plain CE loss')

## Data Setup

In [ ]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF

GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

class RetinopathyDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        path, label = self.records[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomResizedCrop(INPUT_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(INPUT_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

df_all  = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)
df_cv   = df_all[df_all['split'].isin(['train','val'])].copy().reset_index(drop=True)
df_test = df_all[df_all['split'] == 'test'].copy().reset_index(drop=True)
df_cv['cv_idx'] = np.arange(len(df_cv))

def make_records(df):
    return [(row['image_path'], row['grade_int']) for _, row in df.iterrows()]

print(f'CV images : {len(df_cv)}  |  Test images: {len(df_test)}')
print('CV class distribution (images):')
print(df_cv['retinopathy'].value_counts().sort_index())

## Stratified Patient-Level Folds

In [ ]:
# Patient-level stratification on worst grade — identical split to P2B
pat_grade = (
    df_cv.groupby('code')['grade_int']
    .max()
    .reset_index()
    .rename(columns={'grade_int': 'max_grade'})
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pat_grade['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(
    skf.split(pat_grade['code'], pat_grade['max_grade'])
):
    pat_grade.loc[val_idx, 'fold'] = fold_idx

print('Patients per fold:')
print(pat_grade['fold'].value_counts().sort_index())
print('\nClass distribution per fold:')
print(pat_grade.groupby('fold')['max_grade'].value_counts().unstack(fill_value=0))

## WeightedRandomSampler — How It Works

For each training image, we assign a sampling weight = `1 / count_of_its_class`.
PyTorch then draws `num_samples` images **with replacement** from the training set
using these weights, so every class appears approximately equally in each epoch.

```
R0 count = 2200  →  weight = 1/2200 = 0.000455
R3A count =  140  →  weight = 1/140  = 0.007143   (~15× higher)
```

This is computed fresh for each fold (since fold sizes differ slightly).
The loss is plain `CrossEntropyLoss()` — no per-class penalty needed
because the sampler already balances exposure at the data level.

In [ ]:
def make_balanced_sampler(labels):
    """
    Build a WeightedRandomSampler so each class is sampled equally.
    labels: array of integer class labels for the training fold.
    """
    labels      = np.array(labels)
    class_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    # weight per class = 1 / count (classes with 0 samples get weight 0)
    class_weights = np.where(class_counts > 0, 1.0 / class_counts, 0.0)
    # weight per sample = weight of its class
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(sample_weights),
        num_samples = len(labels),   # one 'epoch' = same number of steps as natural data
        replacement = True,          # minority samples are repeated; majority sub-sampled
    )

# Quick sanity check: in a balanced draw, each class should appear ~equally
_dummy_labels = np.array([0]*100 + [1]*50 + [2]*10 + [3]*5)
_sampler = make_balanced_sampler(_dummy_labels)
_drawn   = _dummy_labels[list(_sampler)]
_counts  = np.bincount(_drawn, minlength=4)
print('Sanity check — drawn class counts (should be roughly equal):')
print({CLASSES[i]: int(_counts[i]) for i in range(4)})

## Model & LLRD Optimizer (identical to P2B)

In [ ]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone(device, num_classes=NUM_CLASSES, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True, img_size=INPUT_SIZE,
        num_classes=num_classes, drop_path_rate=0.2,
    )
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt      = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}
    state = model.state_dict()
    drop  = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        del ckpt_model[k]
    model.load_state_dict(ckpt_model, strict=False)
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)
    for param in model.parameters():
        param.requires_grad = True
    model.set_grad_checkpointing(enable=True)
    return model.to(device)

def build_llrd_optimizer(model, base_lr, weight_decay, decay=LLRD_DECAY):
    """
    Layer-wise learning rate decay: head gets base_lr,
    each block group toward input is multiplied by `decay`.
    """
    head_params   = list(model.head.parameters())
    head_ids      = {id(p) for p in head_params}
    blocks        = list(model.blocks)   # 24 transformer blocks
    n_blocks      = len(blocks)
    param_groups  = [{'params': head_params, 'lr': base_lr, 'initial_lr': base_lr}]
    # group blocks in sets of 4; earlier groups get lower LR
    for g_idx in range(n_blocks // 4 - 1, -1, -1):
        lr_g   = base_lr * (decay ** (n_blocks // 4 - g_idx))
        params = []
        for b_idx in range(g_idx * 4, min(g_idx * 4 + 4, n_blocks)):
            params += [p for p in blocks[b_idx].parameters() if id(p) not in head_ids]
        if params:
            param_groups.append({'params': params, 'lr': lr_g, 'initial_lr': lr_g})
    # patch embed + cls token + pos embed
    rest = [p for n, p in model.named_parameters()
            if 'blocks' not in n and 'head' not in n]
    if rest:
        lr_rest = base_lr * (decay ** (n_blocks // 4 + 1))
        param_groups.append({'params': rest, 'lr': lr_rest, 'initial_lr': lr_rest})
    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)

print('Model builder and LLRD optimizer ready.')

## Training Helpers

In [ ]:
class EarlyStopping:
    def __init__(self, patience, model, checkpoint_path):
        self.patience        = patience
        self.best_auroc      = -float('inf')
        self.counter         = 0
        self.checkpoint_path = Path(checkpoint_path)
        torch.save(model.state_dict(), self.checkpoint_path)

    def step(self, auroc, model):
        if auroc != auroc: auroc = 0.0
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model, device):
        model.load_state_dict(torch.load(self.checkpoint_path,
                                         map_location=device, weights_only=True))


def get_lr(epoch, base_lr, min_lr):
    if epoch < WARMUP_EPOCHS:
        return base_lr * (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))


def train_epoch(model, loader, optimizer, criterion, device, scaler, epoch):
    model.train()
    head_lr  = get_lr(epoch, BASE_LR, MIN_LR)
    lr_scale = head_lr / BASE_LR
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * lr_scale

    optimizer.zero_grad()
    total_loss, n_samples, step_count = 0.0, 0, 0

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels  = imgs.to(device), labels.to(device)
        is_last_batch = (i + 1 == len(loader))
        should_step   = ((step_count + 1) % ACCUM_STEPS == 0) or is_last_batch

        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        step_count += 1

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_count = 0

        total_loss += loss.item() * ACCUM_STEPS * len(labels)
        n_samples  += len(labels)

    return total_loss / n_samples, head_lr


@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        with torch.cuda.amp.autocast():
            logits = model(imgs.to(device))
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_probs.append(probs)
    return np.array(all_labels), np.vstack(all_probs)


def compute_metrics(labels, probs):
    preds = probs.argmax(axis=1)
    try:
        auroc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auroc = 0.0
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i].sum() - tp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    return {
        'auroc': auroc,
        'kappa': cohen_kappa_score(labels, preds, weights='quadratic') if len(set(preds)) > 1 else 0.0,
        'macro_sensitivity': float(np.mean(sens)),
        'per_class_sens': sens,
    }

print('Training helpers ready.')

## Cross-Validation Training Loop

In [ ]:
# Plain cross-entropy — no class weights, no focal term
# The sampler handles class balance at the data level
criterion = nn.CrossEntropyLoss()

oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)
test_probs_per_fold = []
fold_results = []

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  [P2E — balanced sampler + plain CE]')
    print(f'{"="*60}')

    val_pats      = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats    = pat_grade[pat_grade['fold'] != fold]['code'].values
    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]
    val_cv_indices = df_fold_val['cv_idx'].values

    # class distribution in this training fold
    train_labels_fold = df_fold_train['grade_int'].values
    counts = np.bincount(train_labels_fold, minlength=NUM_CLASSES)
    print(f'  Train: {len(df_fold_train)} images | Val: {len(df_fold_val)} images')
    print(f'  Train class counts: { {CLASSES[i]: int(counts[i]) for i in range(NUM_CLASSES)} }')

    ds_train = RetinopathyDataset(make_records(df_fold_train), train_transform)
    ds_val   = RetinopathyDataset(make_records(df_fold_val),   eval_transform)
    ds_test  = RetinopathyDataset(make_records(df_test),       eval_transform)

    # Balanced sampler: every class drawn with equal probability per epoch
    sampler = make_balanced_sampler(train_labels_fold)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=4, pin_memory=True, drop_last=False)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    model     = load_backbone(device=DEVICE, seed=SEED + fold)
    optimizer = build_llrd_optimizer(model, BASE_LR, WEIGHT_DECAY, decay=LLRD_DECAY)
    scaler    = torch.cuda.amp.GradScaler()
    ckpt_path = CV_OUTPUT / f'best_fold_{fold}.pth'
    stopper   = EarlyStopping(patience=PATIENCE, model=model, checkpoint_path=ckpt_path)

    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch(model, loader_train, optimizer,
                                       criterion, DEVICE, scaler, epoch)
        val_labels, val_probs = eval_fold(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)
        elapsed = time.time() - t_start
        print(f'  ep {epoch:02d} | lr={cur_lr:.2e} | loss={tr_loss:.4f} | '
              f'AUROC={m["auroc"]:.4f} | MacroSens={m["macro_sensitivity"]:.4f} | '
              f'{elapsed:.0f}s')
        if stopper.step(m['auroc'], model):
            print(f'  Early stop at epoch {epoch} (best AUROC={stopper.best_auroc:.4f})')
            break

    stopper.restore(model, DEVICE)

    oof_labels, oof_probs = eval_fold(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs

    test_labels_fold, test_probs_fold = eval_fold(model, loader_test, DEVICE)
    test_probs_per_fold.append(test_probs_fold)

    m_oof  = compute_metrics(oof_labels, oof_probs)
    m_test = compute_metrics(test_labels_fold, test_probs_fold)
    fold_results.append({'fold': fold, 'oof_auroc': m_oof['auroc'],
                         'test_auroc': m_test['auroc']})
    print(f'  OOF  AUROC={m_oof["auroc"]:.4f}  Kappa={m_oof["kappa"]:.4f}  '
          f'MacroSens={m_oof["macro_sensitivity"]:.4f}')
    print(f'  Test AUROC={m_test["auroc"]:.4f}  Kappa={m_test["kappa"]:.4f}  '
          f'MacroSens={m_test["macro_sensitivity"]:.4f}')

    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',    oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy',   oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy',   test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy',  test_labels_fold)

    del model; torch.cuda.empty_cache()

# Save aggregated arrays
test_probs_ensemble = np.mean(test_probs_per_fold, axis=0)
np.save(CV_OUTPUT / 'oof_probs_all.npy',    oof_probs_all)
np.save(CV_OUTPUT / 'oof_labels_all.npy',   oof_labels_all)
np.save(CV_OUTPUT / 'test_ensemble_probs.npy', test_probs_ensemble)
print('\nAll fold artifacts saved to', CV_OUTPUT)

## Results — P2E vs P2B Comparison

In [ ]:
from sklearn.metrics import roc_auc_score, cohen_kappa_score, confusion_matrix
from sklearn.preprocessing import label_binarize

# ── P2E results ─────────────────────────────────────────────────────────────
def patient_max_pool(codes, probs, labels):
    records = {}
    for c, prob, lbl in zip(codes, probs, labels):
        if c not in records:
            records[c] = {'probs': [], 'worst': 0}
        records[c]['probs'].append(prob)
        records[c]['worst'] = max(records[c]['worst'], int(lbl))
    pt_codes = sorted(records.keys())
    pt_probs, pt_labels = [], []
    for c in pt_codes:
        p = np.stack(records[c]['probs']).max(axis=0)
        pt_probs.append(p / p.sum())
        pt_labels.append(records[c]['worst'])
    return np.array(pt_probs), np.array(pt_labels)

def full_metrics(labels, probs):
    preds   = probs.argmax(axis=1)
    auroc   = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    kappa   = cohen_kappa_score(labels, preds, weights='quadratic')
    acc     = (labels == preds).mean()
    cm      = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens    = []
    for i in range(NUM_CLASSES):
        tp = cm[i,i]; fn = cm[i].sum() - tp
        sens.append(tp/(tp+fn) if (tp+fn)>0 else 0.0)
    return {'auroc': auroc, 'kappa': kappa, 'acc': acc,
            'macro_sens': np.mean(sens), 'per_class_sens': sens}

# test image-level
test_labels = df_test['grade_int'].values
test_codes  = df_test['code'].values

e2e_img_probs = test_probs_ensemble / test_probs_ensemble.sum(axis=1, keepdims=True)
m_img   = full_metrics(test_labels, e2e_img_probs)

# test patient max pool
pt_probs_e, pt_labels_e = patient_max_pool(test_codes, e2e_img_probs, test_labels)
m_pt    = full_metrics(pt_labels_e, pt_probs_e)

# OOF patient max pool
oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')
oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy')
oof_probs  = oof_probs / oof_probs.sum(axis=1, keepdims=True)
oof_codes  = df_cv['code'].values
pt_probs_oof, pt_labels_oof = patient_max_pool(oof_codes, oof_probs, oof_labels)
m_oof_pt = full_metrics(pt_labels_oof, pt_probs_oof)

print('=' * 68)
print('  PHASE 2E — BALANCED SAMPLING RESULTS')
print('=' * 68)
for tag, m in [('OOF Patient Max', m_oof_pt),
               ('Test Image-level', m_img),
               ('Test Patient Max', m_pt)]:
    print(f'\n  {tag}')
    print(f'    AUROC={m["auroc"]:.4f}  Kappa={m["kappa"]:.4f}  Acc={m["acc"]:.4f}')
    print(f'    MacroSens={m["macro_sens"]:.4f}')
    for i, (cn, s) in enumerate(zip(CLASSES, m['per_class_sens'])):
        print(f'    {cn}: sens={s:.4f}')

print('\n' + '=' * 68)
print('  HEAD-TO-HEAD: P2E (Balanced) vs P2B (Focal+Weights)  — Test PtMax')
print('=' * 68)
p2b = {'auroc':0.9396,'kappa':0.8007,'macro_sens':0.6761,
       'per_class_sens':[0.9780,0.8095,0.5833,0.3333]}
p2b_tta = {'auroc':0.9370,'kappa':0.8220,'macro_sens':0.6999,
           'per_class_sens':[0.9780,0.7937,0.5833,0.4444]}

hdr = f'  {"Metric":<18} {"P2B PtMax":>12} {"P2B+TTA":>12} {"P2E PtMax":>12} {"Delta":>10}'
print(hdr)
print('  ' + '-'*66)
for key, label in [('auroc','AUROC'),('kappa','Kappa'),('macro_sens','MacroSens')]:
    v2b  = p2b[key]
    v2bt = p2b_tta[key]
    v2e  = m_pt[key]
    delta = v2e - v2bt
    sign  = '+' if delta >= 0 else ''
    print(f'  {label:<18} {v2b:>12.4f} {v2bt:>12.4f} {v2e:>12.4f}  {sign}{delta:.4f}')
print()
for i, cn in enumerate(CLASSES):
    v2b  = p2b['per_class_sens'][i]
    v2bt = p2b_tta['per_class_sens'][i]
    v2e  = m_pt['per_class_sens'][i]
    delta = v2e - v2bt
    sign  = '+' if delta >= 0 else ''
    print(f'  {cn+" Sensitivity":<18} {v2b:>12.4f} {v2bt:>12.4f} {v2e:>12.4f}  {sign}{delta:.4f}')
print('=' * 68)

## Save Summary

In [ ]:
summary = {
    'phase': 'P2E',
    'description': 'Full fine-tune + WeightedRandomSampler + plain CrossEntropy',
    'test_image_level': m_img,
    'test_patient_max': m_pt,
    'oof_patient_max':  m_oof_pt,
}
# convert numpy floats for JSON serialisation
def to_py(obj):
    if isinstance(obj, dict):  return {k: to_py(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [to_py(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)): return float(obj)
    return obj

with open(CV_OUTPUT / 'phase2e_summary.json', 'w') as f:
    json.dump(to_py(summary), f, indent=2)
print('Summary saved to', CV_OUTPUT / 'phase2e_summary.json')
print('\nExperiment complete. Output directory:', CV_OUTPUT)
print('Files created:')
for p in sorted(CV_OUTPUT.glob('*')): print(' ', p.name)